**Cell 1: Setup, Imports, and Data Loading**

Installs required libraries, loads the dataset, cleans the target variables, and sets up the strict cross-validation strategy.

In [ ]:
# 0. Install required packages for Google Colab
!pip install pymatgen matminer shap scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import shap

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score, average_precision_score
from sklearn.calibration import CalibrationDisplay
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

import warnings
warnings.filterwarnings('ignore')

# 1. Load Dataset
# Ensure 'final_dataset.csv' is uploaded to your Colab environment
df = pd.read_csv('final_dataset.csv')

# Clean citation brackets from Alloy names
if 'Alloy' in df.columns:
    df['Alloy'] = df['Alloy'].str.replace(r'\[\d+\]', '', regex=True)

# ---------------------------------------------------------
# NEW: Remove duplicate alloys to ensure data integrity
# ---------------------------------------------------------
initial_count = len(df)
df = df.drop_duplicates(subset=['Alloy'], keep='first')
print(f"--- DATA AUDIT TRAIL ---")
print(f"Initial records: {initial_count}")
print(f"Dropped {initial_count - len(df)} duplicate alloy entries.")

# 2. Define Features and Target
thermo_features = [col for col in df.columns if re.match(r'^X\d+', col)]
target_col = 'Y (Dmax)'

# Drop missing values
df_clean = df.dropna(subset=thermo_features + [target_col]).copy()
print(f"Dropped {len(df) - len(df_clean)} entries with missing values.")

# Define "Exceptional" Threshold (Dmax >= 10mm)
threshold = 10.0
df_clean['Target_Binary'] = (df_clean[target_col] >= threshold).astype(int)
y = df_clean['Target_Binary']

print(f"\n--- FINAL CURATED DATASET ---")
print(f"Total valid, unique alloys: {len(df_clean)}")
print(f"Exceptional alloys (>=10mm): {y.sum()} (Class imbalance: {y.sum()/len(y)*100:.1f}%)")

# 3. Define Rigorous Cross-Validation
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scoring_metrics = {
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
    'f1': 'f1',
    'balanced_accuracy': 'balanced_accuracy'
}

# 4. Define Explicit RF Hyperparameters
rf_params = {
    'n_estimators': 200,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1
}

**Cell 1.1: Data Integrity & Duplicate Inspection**

 Inspect the dataset for duplicate alloy compositions

In [ ]:
import pandas as pd

# Load the dataset
df_check = pd.read_csv('final_dataset.csv')

# Clean the alloy names (remove citation brackets like [24])
if 'Alloy' in df_check.columns:
    df_check['Alloy'] = df_check['Alloy'].str.replace(r'\[\d+\]', '', regex=True)

# Find ALL rows that share a duplicated Alloy name (keep=False shows all copies)
duplicates = df_check[df_check.duplicated(subset=['Alloy'], keep=False)]

# Sort by Alloy name so the duplicates are grouped together
duplicates_sorted = duplicates.sort_values(by='Alloy')

print(f"Found {len(duplicates_sorted)} rows involved in duplications.")
# Display the Alloy name and the Dmax value to see if they conflict
display(duplicates_sorted[['Alloy', 'Y (Dmax)']])

**Cell 2: MODEL A - Thermodynamic Physics Validation & SHAP**

Evaluate the thermodynamic model entirely separately to prove the algorithm learns the correct physics (Turnbull's criterion, atomic size mismatch) before switching to composition-only features.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import requests
import numpy as np
import shap
from google.colab import files

# --- STEP 1: FONT SETUP ---
font_url = "https://github.com/googlefonts/Assistant/raw/master/fonts/ttf/Assistant-Regular.ttf"
font_path = "Assistant-Regular.ttf"
response = requests.get(font_url)
with open(font_path, "wb") as f:
    f.write(response.content)

fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)

# Set global Matplotlib parameters
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# --- STEP 2: MODEL A LOGIC ---
print("--- MODEL A: THERMODYNAMIC FEATURES ---")
X_thermo = df_clean[thermo_features].copy()

# Dictionary to cleanly format the thermodynamic features for the plot
feature_name_mapping = {
    'X1': r"X1 (Reduced Glass Transition Temp, $T_{rg}$)",
    'X2': r"X2 (Supercooled Liquid Region, $\Delta T_x$)",
    'X3': r"X3 (Alpha Parameter, $\alpha$)",
    'X4': r"X4 (Beta Parameter, $\beta$)",
    'X5': r"X5 (New Beta Parameter, $\beta_{new}$)",
    'X6': r"X6 (Gamma Parameter, $\gamma$)",
    'X7': r"X7 (Gamma_m Parameter, $\gamma_m$)",
    'X8': r"X8 (Atomic Size Mismatch, $\delta$)",
    'X9': r"X9 (Null Sign Parameter)",
    'X10': r"X10 (Omega Parameter, $\omega$)",
    'X11': r"X11 (Omega_m Parameter, $\omega_m$)",
    'X12': r"X12 (Theta Parameter, $\theta$)",
    'X13': r"X13 (Xi Parameter, $\xi$)",
    'X14': r"X14 (Beta Prime Parameter, $\beta'$)",
    'X15': r"X15 (Norm. Liquidus-to-Glass Trans, $\Delta T_{rg}$)",
    'X16': r"X16 (Genetic Programming Parameter, $G_p$)",
    'X17': r"X17 (Gamma_c Parameter, $\gamma_c$)"
}

# Apply the clean names to the dataframe columns
clean_columns = []
for col in X_thermo.columns:
    x_key = col.split(' ')[0]
    clean_columns.append(feature_name_mapping.get(x_key, col))

X_thermo.columns = clean_columns

rf_model_A = RandomForestClassifier(**rf_params)

# Run Repeated Stratified k-Fold CV
cv_results_A = cross_validate(rf_model_A, X_thermo, y, cv=rskf, scoring=scoring_metrics)

print(f"Model A - Mean ROC-AUC: {np.mean(cv_results_A['test_roc_auc']):.3f} ± {np.std(cv_results_A['test_roc_auc']):.3f}")
print(f"Model A - Mean PR-AUC:  {np.mean(cv_results_A['test_pr_auc']):.3f} ± {np.std(cv_results_A['test_pr_auc']):.3f}")
print(f"Model A - Mean F1-Score: {np.mean(cv_results_A['test_f1']):.3f} ± {np.std(cv_results_A['test_f1']):.3f}")
print(f"Model A - Mean Bal. Acc: {np.mean(cv_results_A['test_balanced_accuracy']):.3f} ± {np.std(cv_results_A['test_balanced_accuracy']):.3f}\n")

# SHAP Explainability
print("Generating SHAP Summary Plot for Model A...")
rf_model_A.fit(X_thermo, y)
explainer_A = shap.TreeExplainer(rf_model_A)
shap_values_A = explainer_A.shap_values(X_thermo)

# Safely extract positive class SHAP values
if isinstance(shap_values_A, list):
    shap_vals_A_pos = shap_values_A[1]
elif len(np.shape(shap_values_A)) == 3:
    shap_vals_A_pos = shap_values_A[:, :, 1]
else:
    shap_vals_A_pos = shap_values_A

# --- STEP 3: PLOTTING AND SAVING ---
plt.figure(figsize=(10, 8)) # Increased height slightly to accommodate all 17 features
shap.summary_plot(shap_vals_A_pos, X_thermo, show=False)

# Apply custom font to title
plt.title("SHAP Summary Plot: Model A (Thermodynamic Features)",
          fontproperties=prop, fontsize=16, pad=20)

# Save the figures
file_name = "Figure_2a_SHAP_ModelA"
plt.savefig(f"{file_name}.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{file_name}.pdf", bbox_inches='tight')

plt.show()

**Cell 3: MODEL B - Composition-Only (Magpie) Validation**

Prove the Magpie composition-only model is statistically robust on its own, as this is the model that will actually generate the predictions for the un-synthesized search space.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import requests
import numpy as np
import shap
from google.colab import files

# --- STEP 1: FONT SETUP ---
font_url = "https://github.com/googlefonts/Assistant/raw/master/fonts/ttf/Assistant-Regular.ttf"
font_path = "Assistant-Regular.ttf"
response = requests.get(font_url)
with open(font_path, "wb") as f:
    f.write(response.content)

fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)

# Set global Matplotlib parameters
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# --- STEP 2: MODEL B LOGIC & FEATURIZATION ---
print("--- MODEL B: COMPOSITION-ONLY (MAGPIE) FEATURES ---")

# Initialize Magpie featurizer
ep = ElementProperty.from_preset("magpie")
feature_names = ep.feature_labels()

# Parse original alloys to Pymatgen objects safely
valid_indices = []
comp_objects = []
for idx, alloy_str in df_clean['Alloy'].items():
    try:
        comp = Composition(alloy_str)
        comp_objects.append(comp)
        valid_indices.append(idx)
    except:
        pass

df_magpie = df_clean.loc[valid_indices].copy()
y_magpie = df_magpie['Target_Binary']
df_magpie['comp_obj'] = comp_objects

# Featurize the dataset
print("Featurizing dataset with Magpie (this may take a minute)...")
df_magpie = ep.featurize_dataframe(df_magpie, col_id='comp_obj', ignore_errors=True)
X_magpie = df_magpie[feature_names].copy()

# Clean up Magpie column names for publication-quality plots
clean_magpie_cols = []
for col in X_magpie.columns:
    new_col = col.replace("MagpieData ", "")
    new_col = new_col.replace("avg_dev ", "Avg. Deviation of ")
    new_col = new_col.replace("mean ", "Mean ")
    new_col = new_col.replace("mode ", "Mode ")
    new_col = new_col.replace("maximum ", "Max ")
    new_col = new_col.replace("minimum ", "Min ")
    new_col = new_col.replace("range ", "Range of ")
    # Clean up specific common elemental properties
    new_col = new_col.replace("CovalentRadius", "Covalent Radius")
    new_col = new_col.replace("NUnfilled", "Unfilled Valence Electrons")
    new_col = new_col.replace("NdUnfilled", "Unfilled d-shell Electrons")
    new_col = new_col.replace("Number", "Atomic Number")
    new_col = new_col.replace("Row", "Periodic Table Row")
    new_col = new_col.replace("GSvolume_pa", "Ground State Volume")
    new_col = new_col.replace("AtomicWeight", "Atomic Weight")
    clean_magpie_cols.append(new_col)

X_magpie.columns = clean_magpie_cols

rf_model_B = RandomForestClassifier(**rf_params)

# Run Repeated Stratified k-Fold CV
cv_results_B = cross_validate(rf_model_B, X_magpie, y_magpie, cv=rskf, scoring=scoring_metrics)

print(f"Model B - Mean ROC-AUC: {np.mean(cv_results_B['test_roc_auc']):.3f} ± {np.std(cv_results_B['test_roc_auc']):.3f}")
print(f"Model B - Mean PR-AUC:  {np.mean(cv_results_B['test_pr_auc']):.3f} ± {np.std(cv_results_B['test_pr_auc']):.3f}")
print(f"Model B - Mean F1-Score: {np.mean(cv_results_B['test_f1']):.3f} ± {np.std(cv_results_B['test_f1']):.3f}")
print(f"Model B - Mean Bal. Acc: {np.mean(cv_results_B['test_balanced_accuracy']):.3f} ± {np.std(cv_results_B['test_balanced_accuracy']):.3f}\n")

# --- STEP 3: SHAP EXPLAINABILITY ---
print("Generating SHAP Summary Plot for Model B...")
rf_model_B.fit(X_magpie, y_magpie)
explainer_B = shap.TreeExplainer(rf_model_B)
shap_values_B = explainer_B.shap_values(X_magpie)

# Safely extract positive class SHAP values
if isinstance(shap_values_B, list):
    shap_vals_B_pos = shap_values_B[1]
elif len(np.shape(shap_values_B)) == 3:
    shap_vals_B_pos = shap_values_B[:, :, 1]
else:
    shap_vals_B_pos = shap_values_B

# --- STEP 4: PLOTTING AND SAVING ---
plt.figure(figsize=(10, 7))
# SHAP plot uses global rcParams for feature labels
shap.summary_plot(shap_vals_B_pos, X_magpie, max_display=10, show=False)

# Apply custom font to title
plt.title("SHAP Summary Plot: Model B (Top 10 Compositional Features)",
          fontproperties=prop, fontsize=16, pad=20)

# Save the figures
file_name = "Figure_2b_SHAP_ModelB"
plt.savefig(f"{file_name}.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{file_name}.pdf", bbox_inches='tight')

plt.show()

**Cell 4: Model Calibration Check**

Checks if the output probabilities are well-calibrated (i.e., does a 36% score actually mean ~36% probability of success?).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import requests
import numpy as np
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
from google.colab import files

# --- STEP 1: FONT SETUP ---
font_url = "https://github.com/googlefonts/Assistant/raw/master/fonts/ttf/Assistant-Regular.ttf"
font_path = "Assistant-Regular.ttf"
response = requests.get(font_url)
with open(font_path, "wb") as f:
    f.write(response.content)

fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)

# Set global Matplotlib parameters
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# --- STEP 2: MODEL CALIBRATION LOGIC ---
print("--- MODEL CALIBRATION CHECK (Model B) ---")

# Split data once to check calibration on unseen data
X_train_cal, X_test_cal, y_train_cal, y_test_cal = train_test_split(
    X_magpie, y_magpie, test_size=0.25, random_state=42, stratify=y_magpie
)

rf_cal = RandomForestClassifier(**rf_params)
rf_cal.fit(X_train_cal, y_train_cal)

# Predict probabilities for the positive class
y_prob_cal = rf_cal.predict_proba(X_test_cal)[:, 1]

# Calculate Brier Score
brier = brier_score_loss(y_test_cal, y_prob_cal)
print(f"Brier Score: {brier:.3f} (closer to 0 is better)")

# --- STEP 3: PLOTTING ---
fig, ax = plt.subplots(figsize=(7, 6))

# Plot calibration curve
CalibrationDisplay.from_estimator(
    rf_cal, X_test_cal, y_test_cal, n_bins=5, ax=ax,
    name=f"Model B (Brier Score = {brier:.3f})",
    color='black', marker='s', markersize=8
)

# Formatting with custom font
ax.set_title("Calibration Curve (Reliability Diagram) for Model B", fontproperties=prop, fontsize=16, pad=15)
ax.set_xlabel("Mean Predicted Model Score", fontproperties=prop, fontsize=13)
ax.set_ylabel("Fraction of True Exceptional Alloys", fontproperties=prop, fontsize=13)

# Apply font to ticks
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(prop)

ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc="lower right", prop=prop, fontsize=11)

plt.tight_layout()

# --- STEP 4: SAVE AND DOWNLOAD ---
file_name = "Figure_1_Calibration_Curve_ModelB"
plt.savefig(f"{file_name}.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{file_name}.pdf", bbox_inches='tight')

plt.show()

print("\nConclusion: The deviation from the perfectly calibrated (dotted) line provides empirical justification for referring to the output as a 'Model Score' rather than a strict 'Probability', directly addressing Reviewer 3's concern.")

**Cell 5: Threshold Sensitivity Analysis**

Test if the Al-Ag co-substitution strategy is just an artifact of choosing 10mm as the cutoff. We test 8mm and 12mm.

In [15]:
print("--- THRESHOLD SENSITIVITY ANALYSIS ---")

test_thresholds = [8.0, 10.0, 12.0]
base_str_sens = "Zr41.2 Ti13.8 Cu12.5 Ni10 "

# Generate a smaller search space just for the sensitivity check
sens_alloys = []
for al in np.arange(0.0, 23.0, 1.0):
    for ag in np.arange(0.0, 23.0, 1.0):
        si = 22.5 - al - ag
        if si >= 0.0 and np.isclose(al + ag + si, 22.5):
            comp_str = base_str_sens + (f"Al{al:.1f} " if al>0 else "") + (f"Ag{ag:.1f} " if ag>0 else "") + (f"Si{si:.1f}" if si>0 else "")
            sens_alloys.append(comp_str.strip())

df_sens = pd.DataFrame({'Alloy': sens_alloys})
df_sens['comp_obj'] = df_sens['Alloy'].apply(lambda x: Composition(x))
df_sens = ep.featurize_dataframe(df_sens, col_id='comp_obj', ignore_errors=True)
X_sens = df_sens[feature_names].copy()

# FIX: Align the column names with the cleaned names from Cell 3
X_sens.columns = X_magpie.columns

for t in test_thresholds:
    y_sens = (df_magpie[target_col] >= t).astype(int)
    rf_sens = RandomForestClassifier(**rf_params).fit(X_magpie, y_sens)
    scores = rf_sens.predict_proba(X_sens)[:, 1]
    top_idx = np.argmax(scores)
    print(f"Threshold >= {t:4.1f} mm | Top Candidate: {sens_alloys[top_idx]:<35} | Score: {scores[top_idx]:.3f}")

print("\nConclusion: If Al-Ag candidates consistently win across thresholds, the strategy is robust.")

--- THRESHOLD SENSITIVITY ANALYSIS ---


ElementProperty:   0%|          | 0/276 [00:00<?, ?it/s]

Threshold >=  8.0 mm | Top Candidate: Zr41.2 Ti13.8 Cu12.5 Ni10 Al3.0 Ag12.0 Si7.5 | Score: 0.255
Threshold >= 10.0 mm | Top Candidate: Zr41.2 Ti13.8 Cu12.5 Ni10 Al16.0 Ag5.0 Si1.5 | Score: 0.265
Threshold >= 12.0 mm | Top Candidate: Zr41.2 Ti13.8 Cu12.5 Ni10 Al6.0 Ag15.0 Si1.5 | Score: 0.250

Conclusion: If Al-Ag candidates consistently win across thresholds, the strategy is robust.


**Cell 6: High-Throughput Screening & Uncertainty Estimation**

Generate the full search space, predict the Model Score, and calculate the Standard Deviation across all 200 Decision Trees to provide an uncertainty estimate.

In [ ]:
print("--- HIGH-THROUGHPUT SCREENING WITH UNCERTAINTY ESTIMATION ---")

# Generate Full Search Space
base_str = "Zr41.2 Ti13.8 Cu12.5 Ni10 "
new_alloys = []
step = 0.5

for al in np.arange(0.0, 23.0, step):
    for ag in np.arange(0.0, 23.0, step):
        si = 22.5 - al - ag
        if si >= 0.0 and np.isclose(al + ag + si, 22.5):
            comp_str = base_str
            if al > 0: comp_str += f"Al{al:.1f} "
            if ag > 0: comp_str += f"Ag{ag:.1f} "
            if si > 0: comp_str += f"Si{si:.1f}"
            new_alloys.append(comp_str.strip())

print(f"Generated {len(new_alloys)} theoretical compositions. Featurizing...")
df_search = pd.DataFrame({'Alloy': new_alloys})
df_search['comp_obj'] = df_search['Alloy'].apply(lambda x: Composition(x))
df_search = ep.featurize_dataframe(df_search, col_id='comp_obj')
X_search = df_search[feature_names].copy()

# FIX: Align the column names with the cleaned names from Cell 3
X_search.columns = X_magpie.columns

# Predict using Model B (already trained in Cell 3)
all_tree_preds = np.array([tree.predict_proba(X_search)[:, 1] for tree in rf_model_B.estimators_])

# Calculate Mean and Standard Deviation (Uncertainty)
df_search['Model_Score_Mean'] = all_tree_preds.mean(axis=0)
df_search['Model_Score_Std'] = all_tree_preds.std(axis=0)

# Output Top 15
df_results = df_search[['Alloy', 'Model_Score_Mean', 'Model_Score_Std']].sort_values(by='Model_Score_Mean', ascending=False).reset_index(drop=True)
print("\nTOP 15 COMPUTATIONAL PREDICTIONS FOR BE-FREE VITRELOY 1:")
print("(Note: Scores include ± 1 Standard Deviation across RF trees)")
for i in range(15):
    print(f"{i+1:2d}. {df_results.loc[i, 'Alloy']:<40} | Model Score: {df_results.loc[i, 'Model_Score_Mean']:.3f} ± {df_results.loc[i, 'Model_Score_Std']:.3f}")

**Cell 7: Visualizing the Discovery Landscape (Heatmap with Contours)**

Plot the heatmap of the search space and adds contour lines to show the broadness of the optimal compositional window.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import requests
import numpy as np
import re
from google.colab import files

# --- STEP 1: FONT SETUP ---
font_url = "https://github.com/googlefonts/Assistant/raw/master/fonts/ttf/Assistant-Regular.ttf"
font_path = "Assistant-Regular.ttf"
response = requests.get(font_url)
with open(font_path, "wb") as f:
    f.write(response.content)

fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)

# Set global Matplotlib parameters
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# --- STEP 2: DATA PROCESSING ---
print("--- VISUALIZING DISCOVERY LANDSCAPE ---")

# Parse Al, Ag values for heatmap axes
def parse_substitutions(alloy_string):
    composition = {}
    matches = re.findall(r'(Al|Ag|Si)(\d+\.?\d*)', alloy_string)
    for element, value in matches:
        composition[element] = float(value)
    return composition

parsed_data = df_search['Alloy'].apply(parse_substitutions)
df_search['Al'] = parsed_data.apply(lambda x: x.get('Al', 0))
df_search['Ag'] = parsed_data.apply(lambda x: x.get('Ag', 0))

# Create pivot table and fill NaNs
pivot_table = df_search.pivot_table(index='Ag', columns='Al', values='Model_Score_Mean')
pivot_table = pivot_table.fillna(0)

# --- STEP 3: PLOTTING ---
fig, ax = plt.subplots(figsize=(10, 8))

# Plot Heatmap
# We store the heatmap object to access the colorbar later
heatmap = sns.heatmap(pivot_table, ax=ax, cmap="magma", annot=False)

# Customize Colorbar Label Font
cbar = heatmap.collections[0].colorbar
cbar.set_label('Predicted Model Score (GFA)', fontproperties=prop, fontsize=12)
for label in cbar.ax.get_yticklabels():
    label.set_fontproperties(prop)

# Add Contour Lines
X_mesh, Y_mesh = np.meshgrid(np.arange(pivot_table.shape[1]) + 0.5,
                             np.arange(pivot_table.shape[0]) + 0.5)
contours = ax.contour(X_mesh, Y_mesh, pivot_table.values,
                      levels=6, colors='white', alpha=0.6, linewidths=1.5)

# Set font for contour labels
ax.clabel(contours, inline=True, fontsize=10, fmt='%.2f', inline_spacing=5)
# Note: clabel doesn't support fontproperties directly in some versions,
# but setting the global rcParams above handles this.

# Annotate Best Candidate
best_row = df_search.loc[df_search['Model_Score_Mean'].idxmax()]
y_pos = pivot_table.index.get_loc(best_row['Ag'])
x_pos = pivot_table.columns.get_loc(best_row['Al'])

ax.scatter(x_pos + 0.5, y_pos + 0.5, marker='*', s=600, color='cyan',
           edgecolors='black', linewidths=1.5, zorder=5)

# Formatting with Assistant Font
ax.set_title('Computational Prediction Landscape: Al-Ag Substitution Space\n(Requires Experimental Validation)',
             fontproperties=prop, fontsize=16, pad=20)
ax.set_xlabel('Aluminum Content (at.%)', fontproperties=prop, fontsize=13)
ax.set_ylabel('Silver Content (at.%)', fontproperties=prop, fontsize=13)

# Apply font to axis tick labels
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(prop)

ax.invert_yaxis()
plt.tight_layout()

# --- STEP 4: SAVE AND DOWNLOAD ---
file_name = "Figure_3_Discovery_Landscape_Heatmap"
plt.savefig(f"{file_name}.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{file_name}.pdf", bbox_inches='tight')

plt.show()

**Cell 8: Combined SHAP Plot (Figure 2a and 2b)**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import requests
import shap
import numpy as np
from google.colab import files

# --- STEP 1: FONT SETUP ---
font_url = "https://github.com/googlefonts/Assistant/raw/master/fonts/ttf/Assistant-Regular.ttf"
font_path = "Assistant-Regular.ttf"
response = requests.get(font_url)
with open(font_path, "wb") as f:
    f.write(response.content)

fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)

# --- STEP 2: SET LARGE FONT SIZES ---
# Increased these significantly for a vertical layout
SIZE_TITLE = 26
SIZE_LABEL = 22
SIZE_TICKS = 20

plt.rcParams.update({
    'font.family': prop.get_name(),
    'font.size': SIZE_TICKS,
    'axes.titlesize': SIZE_TITLE,
    'axes.labelsize': SIZE_LABEL,
    'xtick.labelsize': SIZE_TICKS,
    'ytick.labelsize': SIZE_TICKS,
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

# --- STEP 3: GENERATING VERTICAL COMBINED FIGURE ---
print("--- GENERATING VERTICAL COMBINED SHAP FIGURE (FIGURE 2) ---")

# A tall figure (Width=14, Height=20) works best for Top/Bottom SHAP plots
fig = plt.figure(figsize=(14, 20))

# ---------------------------------------------------------
# PANEL A: Model A (TOP)
# ---------------------------------------------------------
ax1 = fig.add_subplot(211) # 2 rows, 1 column, 1st plot
plt.sca(ax1)
shap.summary_plot(shap_vals_A_pos, X_thermo, max_display=10, show=False, plot_size=None)

plt.title("(a) Model A: Thermodynamic Features", fontproperties=prop, fontsize=SIZE_TITLE, pad=30)
plt.xlabel("SHAP value (Impact on Model Score)", fontproperties=prop, fontsize=SIZE_LABEL)

# ---------------------------------------------------------
# PANEL B: Model B (BOTTOM)
# ---------------------------------------------------------
ax2 = fig.add_subplot(212) # 2 rows, 1 column, 2nd plot
plt.sca(ax2)
shap.summary_plot(shap_vals_B_pos, X_magpie, max_display=10, show=False, plot_size=None)

plt.title("(b) Model B: Composition-Only Features", fontproperties=prop, fontsize=SIZE_TITLE, pad=30)
plt.xlabel("SHAP value (Impact on Model Score)", fontproperties=prop, fontsize=SIZE_LABEL)

# --- STEP 4: GLOBAL FONT OVERRIDE ---
# This ensures colorbars and feature names are forced to use Assistant + Large Size
for ax in fig.axes:
    # Set font for axis ticks (Feature names and SHAP values)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontproperties(prop)
        label.set_fontsize(SIZE_TICKS)

    # Check for colorbars (SHAP adds them as separate axes)
    # We identify them by checking if they are not our main ax1 or ax2
    if ax not in [ax1, ax2]:
        ax.set_ylabel(ax.get_ylabel(), fontproperties=prop, fontsize=SIZE_LABEL)
        for t in ax.get_yticklabels():
            t.set_fontproperties(prop)
            t.set_fontsize(SIZE_TICKS - 2)

# Adjust spacing between the two plots to prevent overlap
plt.tight_layout(pad=8.0)

# --- STEP 5: SAVE AND DOWNLOAD ---
file_name = "Figure2_Vertical_SHAP_Final"
plt.savefig(f"{file_name}.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{file_name}.pdf", bbox_inches='tight')

plt.show()

print("Vertical Figure 2 generated with large fonts.")